In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset
from fundus_vessels_toolkit.models.topology.losses import BranchDigraphMiner
from fundus_vessels_toolkit.models.topology.model import BranchDigraphModel, BranchDigraphModelCfg
from fundus_vessels_toolkit.segment_to_graph.tree_simplification import disconnect_crossing, simplify_passing_nodes
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from train import DigraphGNNTrainer, DigraphGNNTrainerConfig

vscode_theme()


/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle_v2.tar.gz")
train_set, val_set, test_set = dataset.split_sets(train_ratio=0.7, val_ratio=0.15, validate_all_graph_versions=False)

Processing...
Done!


In [4]:
cfg = dict(
    gcn=dict(
        bipolar_node=False,
        architecture="InstNorm Conv24x8-DropOut InstNorm Conv48x8-DropOut Conv96x4 Conv48x8 Conv96x4 Conv48x8 Conv96x4 Conv128x4 Conv256x2",
    ),
    img_feature_extractor=dict(type="efficientnet_v2_s", pretrained=True, end_stage=5),
)

n_params = BranchDigraphModel(cfg).model_size()
print("dipole", n_params["graph_features"] + n_params["classif_head"])


cfg["gcn"]["bipolar_node"] = True
n_params = BranchDigraphModel(cfg).model_size()
print("simple", n_params["graph_features"] + n_params["classif_head"])

dipole 6772100
simple 5961156


## Visualize result from pred table


### Load model from checkpoint


In [5]:
# checkpoint = torch.load("GNN-Topo-v1/c2kx8j5h/checkpoints/epoch=239-step=8880.ckpt")
# checkpoint = torch.load("GNN-Topo-v1/ft6svpfg/checkpoints/epoch=179-step=3420.ckpt")
# checkpoint = torch.load("GNN-Topo-v1/96dxz1ex/checkpoints/epoch=99-step=1900.ckpt")

# checkpoint = torch.load("tmp/models/epoch=159-step=2080.ckpt")
# model = checkpoint["hyper_parameters"]["config"]["model"]
# model["gcn"]["architecture"] = "InstNorm Conv64x8-DropOut InstNorm Conv128x8-DropOut Conv128x8 Conv256x4 Conv512x2"

# checkpoint = torch.load("tmp/models/short.ckpt")
# model = checkpoint["hyper_parameters"]["config"]["model"]
# model["gcn"]["legacy_pole_features_ratio"] = True
# checkpoint = torch.load("tmp/Topo-GNN/qukhxch0/checkpoints/epoch=189-step=2470.ckpt")
# checkpoint = torch.load("tmp/Topo-GNN/pw73mdcd/checkpoints/epoch=159-step=2080.ckpt")
checkpoint = torch.load("tmp/Topo-GNN/laes0cil/checkpoints/epoch=199-step=2600.ckpt")
# checkpoint = torch.load("tmp/Topo-GNN/a8whrj6f/checkpoints/epoch=129-step=1690.ckpt")  # Trained on val
# checkpoint = torch.load("tmp/Topo-GNN/7032qrlt/checkpoints/epoch=169-step=2210.ckpt")
# checkpoint = torch.load("tmp/Topo-GNN/l701r6nx/checkpoints/epoch=199-step=2600.ckpt")
checkpoint = torch.load("tmp/Topo-GNN/0jcxsnmt/checkpoints/epoch=289-step=3770.ckpt")


model = checkpoint["hyper_parameters"]["config"]["model"]
model = BranchDigraphModel(model)
model.load_state_dict(
    {
        k[6:]: v
        for k, v in checkpoint["state_dict"].items()
        if k.startswith("model.") and not k.startswith("model.img_feature_extractor.efficient_net")
    }
)
model = model.cuda().eval()

In [ ]:
model.print_model_size()

In [6]:
eval_set = train_set
ID = 153
# CHECK TRANSFER TOPO
# 109 (20051116_55656_0400_PP): [680, 350] AV overlap
# 113 (20051202_37349_0400_PP): [360, 760] invalid sibling affectation

In [6]:
eval_set = val_set
ID = 11

In [7]:
RECHECK = False
if RECHECK:
    ID -= 1
dataset_sample = eval_set.get_sample(ID)
sample_gt, gt_digraph = dataset_sample.to_tensor("gt", return_digraph=True)
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(sample_gt.cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
dir_gt = gt_digraph.branch_dir
print(f"{ID}/{len(eval_set)}: {sample_gt.name}, {dataset_sample.dataset}, {eval_set.samples_info[ID].dataset_type}")
print(
    f"AV: {((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean():.1%} | {((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean():.1%}"
)
print(
    f"Dir: {((out.dir_logit.numpy(force=True)[valid_branch] > 0) == dir_gt[valid_branch]).mean():.1%} | {((pred_dir.numpy(force=True)[valid_branch] > 0) == dir_gt[valid_branch]).mean():.1%}"
)
print(
    f"Parent: {(out.max_parent().numpy(force=True)[valid_branch] == gt_digraph.max_parent()[valid_branch]).mean():.1%} | {(pred_parent.numpy(force=True)[valid_branch] == gt_digraph.max_parent()[valid_branch]).mean():.1%}"
)

m, pred_tree = eval_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
    show_gt_graph=True,
    simplify=not RECHECK,
    version=sample_gt.graph_version,
    branch_label=RECHECK,
    # blood_dir=True,
)
ID += 1
m


153/247: 20060412_60396_0200_PP, MAPLES-DR, train
AV: 99.8% | 99.8%
Dir: 99.5% | 99.5%
Parent: 98.9% | 99.1%


GridBox(children=(HTML(value='<h3 style="text-align: center;">Reference Topology: 20060412_60396_0200_PP</h3>'…

In [ ]:
from jppype import Mosaic

from fundus_toolkits.utils.color import darken_hex
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import assign_av_label_centerline
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees
from fundus_vessels_toolkit.vascular_data_objects.fundus_data import AVLabel

AV_COLORS = {
    AVLabel.BKG: "grey",
    AVLabel.ART: "red",
    AVLabel.VEI: "blue",
    AVLabel.BOTH: "purple",
    AVLabel.UNK: "green",
}

# ID = 10
dataset_sample = eval_set.get_sample(ID, discard_gt_tree=False, load_av_maps=True)
print(ID, dataset_sample.name, dataset_sample.dataset, eval_set.samples_info[ID].dataset_type)
ID += 1


def predict_tree(model, sample, digraph):
    with torch.inference_mode():
        out: BranchDigraphModel.Output = model(sample.cuda())
    pred_parent, pred_dir, pred_av = out.optimal_tree
    pred_parent = pred_parent.numpy(force=True)
    pred_dir = pred_dir.numpy(force=True)
    pred_av = out.av_logit.numpy(force=True)
    pred_fp = (out.fp_logit > 0).numpy(force=True)

    tree = digraph.compute_tree_from_arborescence(
        pred_parent, pred_dir, pred_fp, pred_av, keep_missing_branch=True, assign_av="subtree"
    )
    pred_fp_ext = np.zeros(tree.branch_count, dtype=bool)
    pred_fp_ext[: digraph.branch_count] = pred_fp
    tree.branch_attr.loc[pred_fp_ext, "av"] = AVLabel.BKG
    return tree


def colorize_pred_tree(pred_tree, gt_tree, B: int, simplify=True):
    def next_valid_branch(b_id: int):
        while b_id >= B:
            succs = pred_tree.branch_successors(b_id)
            if len(succs) == 0:
                return None
            b_id = succs[0]
        return b_id

    INVALID = "#37be62"

    # 1. Assign branch colors
    av_pred = pred_tree.branch_attr["av"].to_numpy()
    av_gt = gt_tree.branch_attr["av"].to_numpy()
    for b in pred_tree.branches(np.arange(B)):
        if b.id < B:
            # if gt_av[b.id] == 0:
            #     b.attr["color"] = "white"
            if av_gt[b.id] == 0 or av_gt[b.id] == av_pred[b.id]:
                b.attr["color"] = AV_COLORS[av_pred[b.id]]
            elif av_pred[b.id] == AVLabel.ART:
                b.attr["color"] = "#fc249b"
            elif av_pred[b.id] == AVLabel.VEI:
                b.attr["color"] = "#1c94e3"
            else:
                b.attr["color"] = INVALID

    # 2. Assign dir colors
    for b in pred_tree.branches(np.arange(B)):
        if (
            pred_tree.branch_dirs(b.id) != gt_tree.branch_dirs(b.id)
            and not gt_tree.branch_attr.loc[b.id, "av"] == AVLabel.BKG
        ):
            b.attr["dir_color"] = INVALID
        else:
            b.attr["dir_color"] = b.attr["color"]

    # 3. Check parent validity
    pred_tree.branch_attr["valid_parent"] = True
    fp_pred = av_pred == AVLabel.BKG
    fp_gt = av_gt == AVLabel.BKG
    for b in pred_tree.branches(np.arange(B)):
        # Darken the color of branches without ground truth
        if fp_pred[b.id] or fp_gt[b.id]:
            b.attr["dir_color"] = darken_hex(b.attr["dir_color"], 0.5)
            b.attr["color"] = darken_hex(b.attr["color"], 0.5)
            continue

        parent = pred_tree.branch_tree[b.id]
        while parent >= B:
            parent = pred_tree.branch_tree[parent]
        gt_parent = gt_tree.branch_tree[b.id]
        while gt_parent >= gt_digraph.branch_count:
            gt_parent = gt_tree.branch_tree[gt_parent]
        if gt_parent != parent:
            b.attr["valid_parent"] = False

    # 4. Propagate colors to added branches
    for b in pred_tree.branches(np.arange(B, pred_tree.branch_count)):
        if (next_b := next_valid_branch(b.id)) is not None:
            next_b = pred_tree.branch(next_b)
            if not next_b.attr["valid_parent"]:
                b.attr["color"] = b.attr["dir_color"] = INVALID
            else:
                b.attr["color"] = next_b.attr["color"]
                b.attr["dir_color"] = next_b.attr["dir_color"]
            b.attr["av"] = pred_tree.branch_attr["av"].get(b.id, 0)
        else:
            b.attr["color"] = b.attr["dir_color"] = AV_COLORS[AVLabel.BKG]
            if av_pred is not None:
                b.attr["av"] = AVLabel.BKG

    # 4. Assign node colors
    if simplify:
        disconnect_crossing(pred_tree, inplace=True, fuse_passing_nodes=False)

    pred_tree.node_attr["valid"] = True
    for node in pred_tree.nodes():
        if node.out_degree == 0:
            node.attr["color"] = "grey" if node.in_degree == 0 else node.incoming_branch().attr["color"]
        for b in node.outgoing_branches():
            if not b.attr["valid_parent"]:
                node.attr["color"] = INVALID
                node.attr["valid"] = False
                break
            else:
                node.attr["color"] = b.attr["color"]

    if simplify:
        simplify_passing_nodes(
            pred_tree,
            only_fusable=pred_tree.as_node_ids(pred_tree.node_attr["valid"]),
            min_angle=90,
            with_same_branch_attr=["color", "dir_color"],
            inplace=True,
        )


m = Mosaic(
    (2, 4),
    background=dataset_sample.fundus.image,
    cols_titles=["Ground Truth", "Automorph", "FVT", "VascX"],
    rows_titles=["Raw Graph", "Refined Tree"],
    cell_height=450,
)

for col, v in enumerate(["automorph", "fvt", "vascx"]):
    col += 1
    av_seg = dataset_sample._av_maps[v]
    m[0, col].add_label(av_seg, "AV Seg", colormap=AV_COLORS, opacity=0.2)

    sample_gt, gt_digraph = dataset_sample.to_tensor(v, return_digraph=True)
    gt_tree = gt_digraph.optimize_tree(keep_missing_branch=True, assign_av="subtree")
    fp_gt_ext = np.zeros(gt_tree.branch_count, dtype=bool)
    fp_gt_ext[: gt_digraph.branch_count] = gt_digraph.branch_fp()
    gt_tree.branch_attr.loc[fp_gt_ext, "av"] = AVLabel.BKG

    pred_tree = predict_tree(model, sample_gt, gt_digraph)

    colorize_pred_tree(pred_tree, gt_tree, gt_digraph.branch_count, simplify=True)
    draw_tree(
        pred_tree,
        view=m[1, col],
        branch_color=pred_tree.branch_attr["color"].dropna().to_dict(),
        node_cmap=pred_tree.node_attr["color"].dropna().to_dict(),
        interactive=True,
        bspline_dir=pred_tree.branch_attr["dir_color"].dropna().to_dict(),
        node_dim_roots=False,
    )

    raw_graph = gt_digraph.graph.copy()
    assign_av_label_centerline(raw_graph, av=av_seg, split_av_branch=False, split_high_curvature=False, inplace=True)
    simplify_passing_nodes(raw_graph, min_angle=120, with_same_branch_attr="av", inplace=True)
    draw_graph(raw_graph, view=m[0, col])

m[1, 0].add_image(TreeTopology.av_overlay(dataset_sample.fundus.image, *dataset_sample.target_topologies))
draw_trees(
    (dataset_sample.target_topologies[0].tree, dataset_sample.target_topologies[1].tree),
    view=m[1, 0],
    bspline_dir=True,
    branch_color="av",
)
m

11 085_N FundusAV validation


GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">Ground Truth</h3>'), HTML…

In [ ]:
m[0].goto(gt_digraph.graph.branch(2).midpoint().xy)

In [ ]:
curve = gt_digraph.graph.branch(391).curve().astype(int)
dataset_sample.art_topology.as_dense().fuzzy_skeleton_map[
    *curve.T
]  # - sample.art_topology.as_dense().rank_map[*curve[:-2].T]

In [ ]:
from jppype import Mosaic

from fundus_toolkits.utils.color import darken_hex
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import assign_av_label_centerline
from fundus_vessels_toolkit.utils.jppype import AV_COLORS, draw_graph, draw_tree
from fundus_vessels_toolkit.vascular_data_objects.fundus_data import AVLabel

ID = 15
dataset_sample = eval_set.get_sample(ID, load_av_maps=True)
print(ID, dataset_sample.name, dataset_sample.dataset, eval_set.samples_info[ID].dataset_type)
ID += 1


def predict_tree(model, sample, digraph):
    with torch.inference_mode():
        out: BranchDigraphModel.Output = model(sample.cuda())
    pred_parent, pred_dir, pred_av = out.optimal_tree
    pred_parent = pred_parent.numpy(force=True)
    pred_dir = pred_dir.numpy(force=True)
    pred_av = out.av_logit.numpy(force=True)
    pred_fp = (out.fp_logit > 0).numpy(force=True)

    tree = digraph.compute_tree_from_arborescence(
        pred_parent, pred_dir, pred_fp, pred_av, keep_missing_branch=True, assign_av="subtree"
    )
    pred_fp_ext = np.zeros(tree.branch_count, dtype=bool)
    pred_fp_ext[: digraph.branch_count] = pred_fp
    tree.branch_attr.loc[pred_fp_ext, "av"] = AVLabel.BKG
    return tree


def colorize_pred_tree(pred_tree, gt_tree, B: int, simplify=True):
    def next_valid_branch(b_id: int):
        while b_id >= B:
            succs = pred_tree.branch_successors(b_id)
            if len(succs) == 0:
                return None
            b_id = succs[0]
        return b_id

    INVALID = "#37be62"

    # 1. Assign branch colors
    av_pred = pred_tree.branch_attr["av"].to_numpy()
    av_gt = gt_tree.branch_attr["av"].to_numpy()
    for b in pred_tree.branches(np.arange(B)):
        if b.id < B:
            # if gt_av[b.id] == 0:
            #     b.attr["color"] = "white"
            if av_gt[b.id] == 0 or av_gt[b.id] == av_pred[b.id]:
                b.attr["color"] = AV_COLORS[av_pred[b.id]]
            elif av_pred[b.id] == AVLabel.ART:
                b.attr["color"] = "#fc249b"
            elif av_pred[b.id] == AVLabel.VEI:
                b.attr["color"] = "#1c94e3"
            else:
                b.attr["color"] = INVALID

    # 2. Assign dir colors
    for b in pred_tree.branches(np.arange(B)):
        if (
            pred_tree.branch_dirs(b.id) != gt_tree.branch_dirs(b.id)
            and not gt_tree.branch_attr.loc[b.id, "av"] == AVLabel.BKG
        ):
            b.attr["dir_color"] = INVALID
        else:
            b.attr["dir_color"] = b.attr["color"]

    # 3. Check parent validity
    pred_tree.branch_attr["valid_parent"] = True
    fp_pred = av_pred == AVLabel.BKG
    fp_gt = av_gt == AVLabel.BKG
    for b in pred_tree.branches(np.arange(B)):
        # Darken the color of branches without ground truth
        if fp_pred[b.id] or fp_gt[b.id]:
            b.attr["dir_color"] = darken_hex(b.attr["dir_color"], 0.5)
            b.attr["color"] = darken_hex(b.attr["color"], 0.5)
            continue

        parent = pred_tree.branch_tree[b.id]
        while parent >= B:
            parent = pred_tree.branch_tree[parent]
        gt_parent = gt_tree.branch_tree[b.id]
        while gt_parent >= gt_digraph.branch_count:
            gt_parent = gt_tree.branch_tree[gt_parent]
        if gt_parent != parent:
            b.attr["valid_parent"] = False

    # 4. Propagate colors to added branches
    for b in pred_tree.branches(np.arange(B, pred_tree.branch_count)):
        if (next_b := next_valid_branch(b.id)) is not None:
            next_b = pred_tree.branch(next_b)
            if not next_b.attr["valid_parent"]:
                b.attr["color"] = b.attr["dir_color"] = INVALID
            else:
                b.attr["color"] = next_b.attr["color"]
                b.attr["dir_color"] = next_b.attr["dir_color"]
            b.attr["av"] = pred_tree.branch_attr["av"].get(b.id, 0)
        else:
            b.attr["color"] = b.attr["dir_color"] = AV_COLORS[AVLabel.BKG]
            if av_pred is not None:
                b.attr["av"] = AVLabel.BKG

    # 4. Assign node colors
    if simplify:
        disconnect_crossing(pred_tree, inplace=True, fuse_passing_nodes=False)

    pred_tree.node_attr["valid"] = True
    for node in pred_tree.nodes():
        if node.out_degree == 0:
            node.attr["color"] = "grey" if node.in_degree == 0 else node.incoming_branch().attr["color"]
        for b in node.outgoing_branches():
            if not b.attr["valid_parent"]:
                node.attr["color"] = INVALID
                node.attr["valid"] = False
                break
            else:
                node.attr["color"] = b.attr["color"]

    if simplify:
        simplify_passing_nodes(
            pred_tree,
            only_fusable=pred_tree.as_node_ids(pred_tree.node_attr["valid"]),
            min_angle=90,
            with_same_branch_attr=["color", "dir_color"],
            inplace=True,
        )


m = Mosaic(
    (2, 4),
    background=dataset_sample.fundus.image,
    cols_titles=["Ground Truth", "Automorph", "FVT", "VascX"],
    rows_titles=["Raw Graph", "Refined Tree"],
    cell_height=450,
)

for col, v in enumerate(["automorph", "fvt", "vascx"]):
    col += 1
    av_seg = dataset_sample._av_maps[v]
    m[0, col].add_label(av_seg, "AV Seg", colormap=AV_COLORS, opacity=0.2)

    sample_gt, gt_digraph = dataset_sample.to_tensor(v, return_digraph=True)
    gt_tree = gt_digraph.optimize_tree(keep_missing_branch=True, assign_av="subtree")
    fp_gt_ext = np.zeros(gt_tree.branch_count, dtype=bool)
    fp_gt_ext[: gt_digraph.branch_count] = gt_digraph.branch_fp()
    gt_tree.branch_attr.loc[fp_gt_ext, "av"] = AVLabel.BKG

    pred_tree = predict_tree(model, sample_gt, gt_digraph)

    colorize_pred_tree(pred_tree, gt_tree, gt_digraph.branch_count, simplify=True)
    draw_tree(
        pred_tree,
        view=m[1, col],
        branch_color=pred_tree.branch_attr["color"].dropna().to_dict(),
        node_cmap=pred_tree.node_attr["color"].dropna().to_dict(),
        interactive=True,
        bspline_dir=pred_tree.branch_attr["dir_color"].dropna().to_dict(),
        node_dim_roots=False,
    )

    raw_graph = gt_digraph.graph.copy()
    assign_av_label_centerline(raw_graph, av=av_seg, split_av_branch=False, split_high_curvature=False, inplace=True)
    simplify_passing_nodes(raw_graph, min_angle=120, with_same_branch_attr="av", inplace=True)
    draw_graph(raw_graph, view=m[0, col])

m


In [ ]:
from fundus_vessels_toolkit.models.metrics.tree import DigraphModelOutputInspector

sample = eval_set.get_sample(ID)
sample_tensor = sample.to_tensor("gt", return_digraph=False)
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(sample_tensor.cuda())


In [11]:
inspect = DigraphModelOutputInspector(out, sample)
inspect.metrics()

{'-parent-1tol-acc': tensor(0.9755, device='cuda:0'),
 '-parent-acc': tensor(0.9755, device='cuda:0'),
 '-parent-same-subtree-acc': tensor(0.9790, device='cuda:0'),
 '-parent-same-subtree-mean-dist': tensor(2.6835e-05, device='cuda:0'),
 '-root-sen': tensor(0.8750, device='cuda:0'),
 '-root-spe': tensor(0.7000, device='cuda:0')}

In [ ]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=200, sort_by_p=True).round(3).head(20)

In [ ]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        sample, gt_digraph = eval_set.get(i, version="fvt", return_digraph=True)
        assert VBranchDigraph.has_all_p(gt_digraph) and gt_digraph.graph is not None

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(sample.od_yx.tolist())

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(sample.cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

Parent ACC


In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.argsort(pred_parent_acc)

Dir ACC


In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

AV ACC


In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.argsort(pred_av_acc)

## Test trainer


In [ ]:
eval_set = val_set
ID = 0

In [ ]:
from lightning.pytorch import Trainer
from train import DigraphGNNTrainer
from torch_geometric.loader import DataLoader as PyGDataLoader

trainer_module = DigraphGNNTrainer.load_from_checkpoint("tmp/Topo-GNN/qukhxch0/checkpoints/epoch=189-step=2470.ckpt")
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
    enable_model_summary=False,
    max_epochs=1,
    callbacks=[],
    benchmark=True,
)

In [ ]:
trainer.test(trainer_module, dataloaders=PyGDataLoader(val_set), ckpt_path=None, verbose=True)